# BME 603/606: Computational Methods for Biological Modeling & Simulation
## In-Class Activity: ODEs & Network PageRank

1. **Forward and backward Euler** — implement both integrators for the two-compartment oral absorption model, and see how step size affects accuracy and stability.

2. **Runge–Kutta 4** — code the classical RK4 method and compare its accuracy against Euler for the same step size.

3. **Adjacency and the transition matrix** — build a directed graph from an adjacency matrix, handle dangling nodes, and form the column-stochastic matrix $M$.

4. **PageRank by power iteration** — implement PageRank and rank the nodes of a small network.

5. **Disease-gene prioritization** — run network propagation from a seed set on a small PPI-style graph.

Let's get started!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Plotting defaults
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports done!")

---
## Part 1: ODE Integration of the Oral Absorption Model

### Background: The Two-Compartment Oral Absorption Model

A patient takes a single oral dose. Drug in the gut compartment $A(t)$ is absorbed into plasma $C(t)$ and eliminated:

$$\frac{dA}{dt} = -k_a A(t), \qquad A(0) = F_{\text{dose}}$$

$$\frac{dC}{dt} = k_a A(t) - k_e C(t), \qquad C(0) = 0$$

This is an **initial value problem (IVP)** of the general form

$$y'(t) = f(t, y), \qquad y(0) = y_0, \qquad y = [A, C]^T.$$

Throughout Part 1 we use $F_{\text{dose}} = 500$ mg, $k_a = 1.2\ \mathrm{hr}^{-1}$, $k_e = 0.15\ \mathrm{hr}^{-1}$, and integrate over $[0, 24]$ hr.

In [ ]:
# ── Two-compartment oral absorption model ────────────────────────────────────
ka, ke  = 1.2, 0.15   # 1/h
F_dose  = 500.0       # mg

def f(t, y):
    """Right-hand side of the oral absorption ODE. y = [A, C]."""
    A, C = y
    return np.array([-ka * A, ka * A - ke * C])

T = 24.0
print("Model defined. Integrate over [0, 24] hr.")

### Exercise 1: Forward and Backward Euler

Recall the two first-order integrators from lecture:

**Forward Euler** (explicit, $\mathcal{O}(h)$):

$$y_{i+1} = y_i + h\, f(t_i, y_i)$$

**Backward Euler** (implicit, $\mathcal{O}(h)$):

$$y_{i+1} = y_i + h\, f(t_{i+1}, y_{i+1})$$

Because this system is **linear**, we can write $f(t, y) = J y$ with the constant Jacobian

$$J = \begin{bmatrix} -k_a & 0 \\ k_a & -k_e \end{bmatrix},$$

so backward Euler becomes the linear solve

$$\left(I - h J\right) y_{i+1} = y_i.$$

**Tasks:**
1. Implement `forward_euler(f, y0, t)` and `backward_euler(J, y0, t)`.
2. Run both on the model with $N = 240$ steps ($h = 0.1$ hr) and store the plasma concentration $C(t)$.
3. Run forward Euler again with $N = 2400$ steps ($h = 0.01$ hr). Compare each solution to `solve_ivp` (treat the SciPy solution as ground truth).
4. Plot all three curves on one axes and report the max absolute error of each against the SciPy reference.

> **Hint:** For backward Euler, factor `M = np.eye(2) - h * J` **once** before the loop, then call `np.linalg.solve(M, y[i])` inside.

In [ ]:
def forward_euler(f, y0, t):
    """
    Forward Euler integrator.

    Parameters
    ----------
    f  : callable  — f(t, y) returning dy/dt
    y0 : array-like — initial condition
    t  : 1-D array  — time grid (uniform)

    Returns
    -------
    y : ndarray, shape (len(t), len(y0))
    """
    n = len(t)
    y = np.zeros((n, len(y0)))
    y[0] = y0
    # Your code here
    return y


def backward_euler(J, y0, t):
    """
    Backward Euler integrator for the linear system y' = J y.

    Parameters
    ----------
    J  : ndarray — Jacobian matrix
    y0 : array-like — initial condition
    t  : 1-D array  — time grid (uniform)

    Returns
    -------
    y : ndarray, shape (len(t), len(y0))
    """
    n = len(t)
    h = t[1] - t[0]
    y = np.zeros((n, len(y0)))
    y[0] = y0
    # Your code here — factor (I - h*J) once, then solve each step
    return y


# Reference solution via SciPy
sol_ref = solve_ivp(f, (0, T), [F_dose, 0.0], method='RK45',
                    t_eval=np.linspace(0, T, 500), rtol=1e-8, atol=1e-10)
C_ref_t, C_ref = sol_ref.t, sol_ref.y[1]

# Your code here — build grids, run all three integrators, compute errors, plot

### Exercise 2: Runge–Kutta 4

The classical RK4 method takes four stages per step:

$$\begin{aligned}
k_1 &= h\, f(t_i, y_i) \\
k_2 &= h\, f(t_i + h/2,\ y_i + k_1/2) \\
k_3 &= h\, f(t_i + h/2,\ y_i + k_2/2) \\
k_4 &= h\, f(t_i + h,\ y_i + k_3)
\end{aligned}$$

$$y_{i+1} = y_i + \tfrac{1}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)$$

It is explicit, single-step, and has global error $\mathcal{O}(h^4)$ — at the cost of **four function evaluations per step**.

**Tasks:**
1. Implement `rk4(f, y0, t)`.
2. Run it on the model with $N = 240$ steps ($h = 0.1$ hr) — the *same* step size that made forward Euler inaccurate.
3. Compare the max absolute error of RK4 against the errors you measured for forward Euler at $h = 0.1$ and $h = 0.01$.
4. Plot RK4 and the SciPy reference together. Does RK4 at coarse $h$ beat forward Euler at fine $h$?

In [ ]:
def rk4(f, y0, t):
    """
    Classical 4th-order Runge-Kutta integrator.

    Parameters
    ----------
    f  : callable  — f(t, y) returning dy/dt
    y0 : array-like — initial condition
    t  : 1-D array  — time grid (uniform)

    Returns
    -------
    y : ndarray, shape (len(t), len(y0))
    """
    n = len(t)
    h = t[1] - t[0]
    y = np.zeros((n, len(y0)))
    y[0] = y0
    # Your code here — four stages per step
    return y


# Your code here — run RK4 at h = 0.1, compare to Euler and reference

---
## Part 2: PageRank on a Small Network

### Background: The Random Surfer Model

A random surfer moves on a directed graph $G = (V, E)$:

- with probability $\alpha$ (the **damping factor**), follow a uniformly random outgoing edge;
- with probability $1 - \alpha$, **teleport** to a uniformly random node.

**PageRank** is the stationary distribution of this walk: $r_i$ is the long-run fraction of time the surfer spends at node $i$.

### Mathematical Formulation

Let $A$ be the adjacency matrix ($A_{ij} = 1$ if edge $j \to i$ exists). Normalize each column by the out-degree

$$M_{ij} = \frac{A_{ij}}{d_j^{\text{out}}}$$

so that $M$ is column-stochastic. **Dangling nodes** (columns with $d_j^{\text{out}} = 0$) break column-stochasticity; fix them by replacing the zero column with $1/n$:

$$\hat{M} = M + \frac{1}{n}\mathbf{1}\,\mathbf{d}^\top, \qquad d_j = 1 \text{ if } d_j^{\text{out}} = 0.$$

The **Google matrix** adds teleportation:

$$G = \alpha \hat{M} + \frac{1-\alpha}{n}\mathbf{1}\mathbf{1}^\top,$$

and the PageRank vector satisfies

$$\mathbf{r} = \alpha \hat{M}\mathbf{r} + \frac{1-\alpha}{n}\mathbf{1}, \qquad \|\mathbf{r}\|_1 = 1.$$

By **Perron–Frobenius**, $G$ has a unique positive dominant eigenvector (eigenvalue $\lambda_1 = 1$).

### Exercise 3: Build the Transition Matrix

Use this adjacency matrix for a 4-node graph:

$$A = \begin{bmatrix}
0 & 0 & 1 & 0 \\
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 1 & 0 & 1
\end{bmatrix}$$

(Here $A_{ij} = 1$ means edge $j \to i$ exists.)

**Tasks:**
1. Compute the out-degree vector `d = A.sum(axis=0)` and identify any dangling nodes.
2. Build the column-stochastic matrix `M = A / d` (with a divide-by-zero guard), then set the dangling columns to `1/n`.
3. Verify: `M.sum(axis=0)` should equal `np.ones(n)`.
4. Print `M` and inspect the structure.

> **Hint:** `d[d == 0] = 1` before dividing, then handle dangling columns separately.

In [ ]:
A = np.array([[0, 0, 1, 0],
              [1, 0, 0, 0],
              [0, 1, 0, 0],
              [0, 1, 0, 1]], dtype=float)

n = A.shape[0]

# Your code here — out-degrees, dangling mask, column-stochastic M

### Exercise 4: PageRank by Power Iteration

The power iteration is:

1. Initialize $\mathbf{r}^{(0)} = \tfrac{1}{n}\mathbf{1}$.
2. Iterate $\mathbf{r}^{(k+1)} = \alpha \hat{M}\mathbf{r}^{(k)} + \tfrac{1-\alpha}{n}\mathbf{1}$.
3. Stop when $\|\mathbf{r}^{(k+1)} - \mathbf{r}^{(k)}\|_1 < \epsilon$.

**Tasks:**
1. Implement `pagerank(M, alpha=0.85, tol=1e-6, max_iter=100)`.
2. Run it on the matrix from Exercise 3.
3. Print the converged PageRank vector, and report which node ranks highest.
4. Try $\alpha = 0.5$ and $\alpha = 0.99$. How does the ranking change? Why?

In [ ]:
def pagerank(M, alpha=0.85, tol=1e-6, max_iter=100):
    """
    PageRank via power iteration.

    Parameters
    ----------
    M        : ndarray — column-stochastic matrix (dangling nodes already handled)
    alpha    : float   — damping factor
    tol      : float   — convergence tolerance on the L1 norm
    max_iter : int     — maximum number of iterations

    Returns
    -------
    r : ndarray, shape (n,) — PageRank vector
    """
    n = M.shape[0]
    r = np.ones(n) / n
    # Your code here
    return r


# Your code here — run pagerank, print the vector, report the top node

### Exercise 5: Disease-Gene Prioritization by Network Propagation

**Motivation.** Given a disease, we want to rank all genes in a protein–protein interaction (PPI) network by how likely they are to be causally involved. We start from a small **seed set** $S$ of known disease-associated genes and propagate their scores through the network.

**Method** (Vanunu et al., 2010). Initialize

$$f_i^{(0)} = \frac{1}{|S|} \text{ if } i \in S, \quad 0 \text{ otherwise},$$

then iterate

$$\mathbf{f}^{(k+1)} = \alpha\, M \mathbf{f}^{(k)} + (1 - \alpha)\, \mathbf{f}^{(0)}.$$

The second term **pins** the seed scores, so the propagation does not wash them out — this is the key difference from plain PageRank.

**Tasks:**
1. Use the same 4-node matrix `M` from Exercise 3, but treat node 0 as the only seed: $S = \{0\}$.
2. Implement `network_propagation(M, f0, alpha=0.85, tol=1e-6, max_iter=100)`.
3. Run it, print the converged scores, and rank the nodes.
4. Compare the ranking to the PageRank ranking from Exercise 4. Which non-seed node is prioritized most, and why does that make sense given the graph structure?

> **Hint:** Reuse `pagerank`'s iteration structure, but replace the uniform teleportation term $(1-\alpha)/n \cdot \mathbf{1}$ with $(1-\alpha)\mathbf{f}^{(0)}$.

In [ ]:
def network_propagation(M, f0, alpha=0.85, tol=1e-6, max_iter=100):
    """
    Network propagation with a pinned seed vector.

    Parameters
    ----------
    M        : ndarray — column-stochastic matrix
    f0       : ndarray — initial (seed) score vector
    alpha    : float   — damping factor
    tol      : float   — convergence tolerance
    max_iter : int     — maximum iterations

    Returns
    -------
    f : ndarray, shape (n,) — propagated score vector
    """
    f = f0.copy()
    # Your code here
    return f


# Seed set: node 0
f0 = np.zeros(n)
f0[0] = 1.0

# Your code here — run network propagation, print, and compare to PageRank

---
## Further Exploration

### Stiffness and Implicit Methods

Forward Euler requires $h \lesssim 2/k_a$ for this model. When rate constants span many orders of magnitude (e.g. a fast binding reaction coupled to slow gene expression), the stability limit becomes prohibitive and **implicit methods** like backward Euler become essential — they are unconditionally stable for linear problems. The price is a linear (or nonlinear) solve at every step.

### Adaptive Step Sizes

`solve_ivp(method='RK45')` uses an embedded Runge–Kutta pair to estimate local error and adapt $h$ automatically. Try varying `rtol` and `atol` and watch the number of function evaluations change via `sol.nfev`.

### PageRank in Practice

Real web graphs have billions of nodes. Power iteration still works because the matrix is **sparse** — each iteration is one sparse matrix-vector product. Modern search engines use many refinements, but the core idea is the same fixed-point equation $\mathbf{r} = \alpha \hat{M}\mathbf{r} + \tfrac{1-\alpha}{n}\mathbf{1}$.

### References & Resources

- [Forward Euler — Wikipedia](https://en.wikipedia.org/wiki/Euler_method)
- [Backward Euler — Wikipedia](https://en.wikipedia.org/wiki/Backward_Euler_method)
- [Runge–Kutta methods — Wikipedia](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods)
- [PageRank — Wikipedia](https://en.wikipedia.org/wiki/PageRank)
- [Perron–Frobenius theorem — Wikipedia](https://en.wikipedia.org/wiki/Perron%E2%80%93Frobenius_theorem)
- [SciPy `solve_ivp` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)